# 🚀 Channel Micro-POV — Unattended Render Farm (Kaggle GPU T4 x2 + Blender 5.2.2)

Diadaptasi dari `skills/blender-render-farm/notebooks/kaggle_render.ipynb` agar kamu bisa klik **Save Version → Save & Run All (Commit)**, tutup browser, lalu unduh **`final_ep001.mp4`** langsung dari tab **Output**!

### Pengaturan Kaggle (Panel Kanan — Wajib):
1. **Accelerator:** `GPU T4 x2` (2× NVIDIA Tesla T4 aktif paralel via OptiX/CUDA + OpenImageDenoise)
2. **Internet:** `On`
3. **Add Input (Disarankan):** Tambahkan Kaggle Dataset privat berisi `family_ready.blend` (dan opsional `scene_01.wav .. scene_08.wav` dari Drive agar tidak perlu regenerasi TTS).
4. **Add-ons → Secrets (Opsional):**
   - `GEMINI_API_KEY` (hanya jika VO belum ada di Dataset Input)
   - `GITHUB_TOKEN` (hanya jika repo GitHub bersifat private)
5. Klik **Save Version → Save & Run All (Commit)** — aman ditinggal tutup browser hingga `final_ep001.mp4` selesai!

In [ ]:
# ---- PENGATURAN JOB RENDER (UBAH SESUAI KEBUTUHAN) ----
REPO_URL       = "https://github.com/ruangcermat/channel-micro-pov.git"
REPO_BRANCH    = "main"
EPISODE_SPEC   = "04_episodes/ep001_kecoak_dapur/ep001_spec.json"
RENDER_MODE    = "final"      # "storyboard" | "preview" (540x960) | "final" (1080x1920)
RENDER_ENGINE  = "CYCLES"     # "CYCLES" (2xT4 GPU + OIDN) | "EEVEE" (super cepat 2-6 dtk/frame)
SINGLE_SCENE   = 0            # 0 = render semua scene; 1..8 = render 1 scene saja
TTS_VOICE      = "Puck"
VO_SPEED       = 1.08
SKIP_EXISTING_VO = True

# Opsional: jika family_ready.blend belum dimasukkan ke Kaggle Input Dataset,
# isi Google Drive File ID (yang sudah di-share "Anyone with the link") untuk auto-download via gdown:
FAMILY_BLEND_GDRIVE_ID = ""
ZIP_FRAMES_OUTPUT = False     # True jika ingin ikut menyimpan frames.zip di Output tab

In [ ]:
!nvidia-smi --query-gpu=index,name,memory.total,driver_version --format=csv

In [ ]:
# ---- 1. SETUP SECRETS, DEPENDENSI & BLENDER 5.2.2 LTS ----
import glob
import os
import shutil
import subprocess
from pathlib import Path

def run_step(cmd):
    """Jalankan perintah dengan output real-time penuh (stdout + stderr)."""
    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end="", flush=True)
    rc = proc.wait()
    if rc != 0:
        raise subprocess.CalledProcessError(rc, cmd)

# Muat Kaggle Secrets opsional
GITHUB_TOKEN = ""
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    for key in ("GEMINI_API_KEY", "GITHUB_TOKEN"):
        try:
            val = secrets.get_secret(key)
            if val and val.strip():
                if key == "GITHUB_TOKEN":
                    GITHUB_TOKEN = val.strip()
                else:
                    os.environ[key] = val.strip()
                print(f"✅ Loaded Kaggle Secret: {key}")
        except Exception:
            pass
except Exception:
    pass

ROOT_WORK = Path("/kaggle/working")
REPO_DIR = ROOT_WORK / "channel-micro-pov"
ASSETS_DIR = ROOT_WORK / "assets_cache"
EPISODE_SLUG = Path(EPISODE_SPEC).parent.name
WORK_DIR = ROOT_WORK / "ep_work" / EPISODE_SLUG
ASSETS_DIR.mkdir(parents=True, exist_ok=True)
(ASSETS_DIR / "models_3d").mkdir(parents=True, exist_ok=True)
(WORK_DIR / "raw_render" / "vo").mkdir(parents=True, exist_ok=True)

# Clone/Update GitHub repo
AUTH_REPO_URL = REPO_URL
if GITHUB_TOKEN and REPO_URL.startswith("https://"):
    AUTH_REPO_URL = REPO_URL.replace("https://", f"https://{GITHUB_TOKEN}@", 1)

if (REPO_DIR / ".git").exists():
    run_step(["git", "-C", str(REPO_DIR), "fetch", "origin"])
    run_step(["git", "-C", str(REPO_DIR), "checkout", REPO_BRANCH])
    run_step(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", REPO_BRANCH])
else:
    run_step(["git", "clone", "-b", REPO_BRANCH, AUTH_REPO_URL, str(REPO_DIR)])

# Install system libraries (GL/X11/FFmpeg/Fonts) & Python packages
run_step(["bash", "-lc", "apt-get -qq update > /dev/null && apt-get -qq install -y ffmpeg fonts-dejavu-core libxrender1 libxxf86vm1 libxfixes3 libxi6 libxkbcommon0 libsm6 libgl1 libegl1 libglu1-mesa > /dev/null"])
run_step(["pip", "install", "-q", "--upgrade", "-r", str(REPO_DIR / "03_pipeline/requirements.txt"), "gdown"])

# Install Blender 5.2.2 LTS Linux Headless
BLENDER_DIR = ROOT_WORK / "blender52"
BLENDER_BIN = BLENDER_DIR / "blender"
if not BLENDER_BIN.exists():
    tar_path = "/tmp/blender-5.2.2-linux-x64.tar.xz"
    run_step(["wget", "-q", "https://download.blender.org/release/Blender5.2/blender-5.2.2-linux-x64.tar.xz", "-O", tar_path])
    BLENDER_DIR.mkdir(parents=True, exist_ok=True)
    run_step(["tar", "-xf", tar_path, "-C", str(BLENDER_DIR), "--strip-components=1"])
    os.remove(tar_path)
run_step([str(BLENDER_BIN), "--version"])

In [ ]:
# ---- 2. AUTO-DISCOVERY MPFB2 family_ready.blend & CACHED VO DARI /kaggle/input ----
FAMILY_CACHE = ASSETS_DIR / "models_3d" / "family_ready.blend"

# 2a. Gabungkan parts (*.blend.part*) jika di-upload terpecah, atau salin langsung family_ready.blend
parts = sorted(
    glob.glob("/kaggle/input/**/family_ready.blend.part*", recursive=True),
    key=lambda p: int(p.rsplit("part", 1)[1]),
)
if parts and not FAMILY_CACHE.exists():
    with open(FAMILY_CACHE, "wb") as out_fh:
        for p in parts:
            out_fh.write(open(p, "rb").read())
    print(f"✅ Joined {len(parts)} parts -> {FAMILY_CACHE}")

if not FAMILY_CACHE.exists():
    found_blends = sorted(glob.glob("/kaggle/input/**/family_ready.blend", recursive=True))
    if found_blends:
        shutil.copy2(found_blends[0], FAMILY_CACHE)
        print(f"✅ Copied family_ready.blend from Kaggle Input -> {FAMILY_CACHE}")
    elif FAMILY_BLEND_GDRIVE_ID:
        run_step(["gdown", "--id", FAMILY_BLEND_GDRIVE_ID, "-O", str(FAMILY_CACHE)])

assert FAMILY_CACHE.exists(), (
    "family_ready.blend tidak ditemukan! Tambahkan Kaggle Dataset berisi family_ready.blend "
    "via Add Input atau isi FAMILY_BLEND_GDRIVE_ID di sel pengaturan."
)
os.environ["MICRO_POV_FAMILY_BLEND"] = str(FAMILY_CACHE)

# 2b. Auto-copy cached VO WAVs (scene_01.wav .. scene_08.wav) dari Kaggle Input jika tersedia
vo_dest = WORK_DIR / "raw_render" / "vo"
copied_vo = 0
for wav in sorted(glob.glob("/kaggle/input/**/scene_*.wav", recursive=True)):
    dst = vo_dest / os.path.basename(wav)
    if not dst.exists():
        shutil.copy2(wav, dst)
        copied_vo += 1
print(f"✅ Cached VO WAVs ditemukan & disalin dari Kaggle Input: {copied_vo} file")

In [ ]:
# ---- 3. FETCH 15 POLY HAVEN CC0 PROPS + TIMING AUDIO-FIRST + SFX/BGM ----
SPEC_PATH = REPO_DIR / EPISODE_SPEC

run_step([
    "python3", str(REPO_DIR / "03_pipeline/scripts/fetch_assets.py"),
    "--spec", str(SPEC_PATH),
    "--assets-dir", str(ASSETS_DIR),
])

tts_cmd = [
    "python3", str(REPO_DIR / "03_pipeline/scripts/generate_tts.py"),
    "--spec", str(SPEC_PATH),
    "--work-dir", str(WORK_DIR),
    "--voice", TTS_VOICE,
    "--fps", "24",
    "--pad-sec", "0.06",
    "--vo-speed", str(VO_SPEED),
]
if SKIP_EXISTING_VO:
    tts_cmd.append("--skip-existing")
run_step(tts_cmd)

run_step([
    "python3", str(REPO_DIR / "03_pipeline/scripts/prepare_audio.py"),
    "--spec", str(SPEC_PATH),
    "--assets-dir", str(ASSETS_DIR),
])

In [ ]:
# ---- 4. RENDER 3D BLENDER 5.2.2 (2xT4 GPU + OIDN + post.py + AUTO-RESUME) ----
render_cmd = [
    str(BLENDER_BIN),
    "--background",
    "--python", str(REPO_DIR / "03_pipeline/scripts/build_and_render.py"),
    "--",
    "--spec", str(SPEC_PATH),
    "--timing", str(WORK_DIR / "timing_manifest.json"),
    "--assets-dir", str(ASSETS_DIR),
    "--work-dir", str(WORK_DIR),
    "--mode", RENDER_MODE,
    "--engine", RENDER_ENGINE,
]
if SINGLE_SCENE > 0:
    render_cmd.extend(["--scene", str(SINGLE_SCENE)])

run_step(render_cmd)

In [ ]:
# ---- 5. SUBTITLE .ASS 9:16 + MIXING FINAL (-14 LUFS) + EKSPOR KE /kaggle/working ----
import json

run_step([
    "python3", str(REPO_DIR / "03_pipeline/scripts/compose_final.py"),
    "--spec", str(SPEC_PATH),
    "--timing", str(WORK_DIR / "timing_manifest.json"),
    "--assets-dir", str(ASSETS_DIR),
    "--work-dir", str(WORK_DIR),
    "--mode", RENDER_MODE,
])

manifest = json.loads((WORK_DIR / "timing_manifest.json").read_text(encoding="utf-8"))
ep_id = manifest["episode_id"].lower()
if RENDER_MODE == "final":
    out_name = f"final_{ep_id}.mp4"
elif RENDER_MODE == "storyboard":
    out_name = f"animatic_{ep_id}.mp4"
else:
    out_name = f"preview_{ep_id}.mp4"

src_mp4 = WORK_DIR / out_name
dst_mp4 = ROOT_WORK / out_name
shutil.copy2(src_mp4, dst_mp4)
print(f"🎉 OUTPUT READY: {dst_mp4} ({dst_mp4.stat().st_size / 1e6:.2f} MB)")

for png_name in ("storyboard_sheet.png", "storyboard_grid.png"):
    src_p = WORK_DIR / png_name
    if src_p.exists():
        shutil.copy2(src_p, ROOT_WORK / png_name)

if ZIP_FRAMES_OUTPUT:
    scenes_dir = WORK_DIR / "raw_render" / f"scenes_{RENDER_MODE}"
    if scenes_dir.exists():
        run_step(["zip", "-qr", str(ROOT_WORK / f"frames_{ep_id}.zip"), str(scenes_dir)])

In [ ]:
# ---- 6. BERSIHKAN CACHE BESAR AGAR TAB OUTPUT KAGGLE HANYA MENAMPILKAN HASIL ----
!rm -rf /kaggle/working/blender52 /kaggle/working/assets_cache /kaggle/working/channel-micro-pov /kaggle/working/ep_work
!ls -lh /kaggle/working